# PDF 문서에서 이미지 TEXT / IMAGE / TABLE 읽기

In [2]:
!pip install langchain langchain-community pymupdf pytesseract pillow transformers

  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached pymupdf-1.28.0-cp310-abi3-manylinux_2_28_x86_64.whl.metadata (26 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [24]:
from langchain_community.document_loaders import PyMuPDFLoader
import fitz # pdf파일 처리해줄수 있게 해주는 모듈
from PIL import Image
import pytesseract # ocr(이미지 안의 글씨 읽을때 사용)
import os
import io
import matplotlib.pyplot as plt

- Document - LangChain 의 기본 문서 객체입니다.

  [속성]
  - page_content: 문서의 내용을 나타내는 문열입니다.
  - metadata: 문서의 메타데이터를 나타내는 딕셔너리입니다.

In [18]:
# 텍스트 읽기 1 : PyMuPDFLoader 사용
pdf_path = "aiethics.pdf"
loader = PyMuPDFLoader(pdf_path)
documents = loader.load() # pdf의 각 페이지가 Document객체로 변환.
print(documents)

# pdf의 텍스트만 출력
full_text = '\n\n'.join([doc.page_content.strip() for doc in documents if doc.page_content.strip()])
print (full_text[:100])

[Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign CC 13.1 (Macintosh)', 'creationdate': '2024-03-07T15:09:14+09:00', 'source': 'aiethics.pdf', 'file_path': 'aiethics.pdf', 'total_pages': 16, 'format': 'PDF 1.3', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-03-07T15:09:28+09:00', 'trapped': '', 'modDate': "D:20240307150928+09'00'", 'creationDate': "D:20240307150914+09'00'", 'page': 0}, page_content='전북교육 2023-466\n발\n간\n등\n록\n번\n호\n1'), Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign CC 13.1 (Macintosh)', 'creationdate': '2024-03-07T15:09:14+09:00', 'source': 'aiethics.pdf', 'file_path': 'aiethics.pdf', 'total_pages': 16, 'format': 'PDF 1.3', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-03-07T15:09:28+09:00', 'trapped': '', 'modDate': "D:20240307150928+09'00'", 'creationDate': "D:20240307150914+09'00'", 'page': 1}, page_content='똑디(똑똑한 디지털 도우미)와 함께하는 생성형 AI·

In [19]:
# 텍스트 읽기 2) fitz
from langchain_core.documents import Document

doc = fitz.open('aiethics.pdf')
print(doc)

documents = []
for i, page in enumerate(doc):
  text = page.get_text()
  metadata = {
      'source':pdf_path,
      'page':i + 1,
      'total_pages': len(doc),
      'author':doc.metadata.get('author',''),
      'title':doc.metadata.get('title',''),
      'producer':doc.metadata.get('producer','')
  }
  documents.append(Document(page_content=text.strip(), metadata=metadata))

print(f'총 문서 수 : {len(documents)}')
print(f'첫번째 페이지 : {documents[0]}')
full_text = '\n\n'.join([doc.page_content.strip() for doc in documents if doc.page_content.strip()])
print (full_text[100:200])

Document('aiethics.pdf')
총 문서 수 : 16
첫번째 페이지 : page_content='전북교육 2023-466
발
간
등
록
번
호
1' metadata={'source': 'aiethics.pdf', 'page': 1, 'total_pages': 16, 'author': '', 'title': '', 'producer': 'Adobe PDF Library 15.0'}
AI 란  _04
2. 생성형 AI 사용 연령 제한  _06
3. 생성형 AI 사용 시 개인정보보호 및 존중  _06 
4. 저작권을 침해하지 않는 생성형 AI 일러스트레이션  _


In [41]:
# 이미지 읽기
doc = fitz.open(pdf_path)
ocr_results = [] # OCR결과를 저장할 list 생성

for page_num, page in enumerate(doc):
  images = page.get_images(full=True)
  if not images:
    continue
  # print(images)

  for i, img in enumerate(images):
    xref = img[0] # 이미지의 참조 id
    base_image = doc.extract_image(xref)
    image_bytes = base_image["image"] # 이미지의 raw byte data를 가져옴
    image = Image.open(io.BytesIO(image_bytes))

    # 문서에서 읽은 이미지를 각각 파일로 디렉토리에 저장
    save_dirs = 'imgs'
    os.makedirs(save_dirs, exist_ok=True)
    image_filename = f'page_{page_num + 1}_image_{i + 1}.jpeg'
    image_path = os.path.join(save_dirs, image_filename)
    image.save(image_path)
    # print(f'Image 저장 완료 : {image_path}')

    # OCR(광학문자인식)을 적용
    # 이미지를 분석 후 텍스트 + 위치 + 신뢰도(confidence)등의 정보를 dict타입으로 반환
    data = pytesseract.image_to_data(image, output_type=pytesseract.Output.DICT)
    # print('data :', data)
    # print(data['text'])

    # 신뢰도가 60이상인 단어만 골라(잡음제거) 문자열로 결합
    extracted_text = ' '.join([text for j, text in enumerate(data['text']) if int(data['conf'][j]) >= 60])
    # print(extracted_text)
    ocr_results.append(
        {
            'page':page_num + 1,
            'text':extracted_text,
            'image_indes':i + 1
        }
    )
    # 이미지 보기
    plt.figure(figsize=(3,3))
    plt.imshow(image)
    for j in range(len(data['text'])):
      if int(data['conf'][j]) >= 60: # OCR text중 신뢰도 60 이상인것만 시각화
        # text에 빨간 박스 그리기
        (x, y, w, h) = (data['left'][j], data['top'][j], data['width'][j], data['height'][j])
        plt.gca().add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor='red', facecolor='none',linewidth=1))
    plt.axis('off')
    plt.title(f"ocr result on Page {page_num + 1}, image{i + 1}")
    plt.show()

print(ocr_results[:2])

Output hidden; open in https://colab.research.google.com to view.